# Web Crawler Demo: CNN

This notebook demonstrates the WebCrawler Spider by crawling CNN.com and analyzing the results.

In [ ]:
import json
import logging
from collections import Counter

import pandas as pd

from WebCrawler import Serializers, Spider

# Configure logging to see crawler activity
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

## Set up and run the Spider

We'll crawl CNN with a depth of 1 (homepage + links from the homepage) to keep it manageable.

In [ ]:
# Create Spider instance
start_url = "https://www.cnn.com"
max_depth = 1  # Keep it shallow to avoid excessive requests

spider = Spider(start_url=start_url, max_depth=max_depth, debug=True)

print(f"Starting crawl of {start_url} (max depth: {max_depth})...\n")

# Run the async crawler (await works in Jupyter)
documents = await spider.run_async()

print(f"\nCrawl complete! Visited {len(documents)} pages.")

## Analyze Results

In [ ]:
# Display page titles and link counts
print(f"{'URL':<60} {'Title':<40} {'Links':<8}")
print("-" * 110)

for doc in documents:
    url_short = doc.url[:59] if len(doc.url) > 59 else doc.url
    title_short = doc.title[:39] if len(doc.title) > 39 else doc.title
    num_links = len(doc.links)
    print(f"{url_short:<60} {title_short:<40} {num_links:<8}")

## Statistics

In [ ]:
# Aggregate statistics
total_links = sum(len(doc.links) for doc in documents)
total_internal = sum(len(doc.internal_links) for doc in documents)
total_external = sum(len(doc.external_links) for doc in documents)

print(f"Total pages crawled: {len(documents)}")
print(f"Total links found: {total_links}")
print(f"  - Internal: {total_internal}")
print(f"  - External: {total_external}")
print(f"\nAverage links per page: {total_links / len(documents):.1f}")

## Sample External Links from Homepage

In [ ]:
# Show external links from the first page (homepage)
if documents:
    homepage = documents[0]
    print(f"External links from {homepage.url}:\n")
    for link in homepage.external_links[:10]:  # Show first 10
        print(f"  • {link.url}")
    if len(homepage.external_links) > 10:
        print(f"  ... and {len(homepage.external_links) - 10} more")

## Sample Internal Links from Homepage

In [ ]:
# Show internal links from the first page (homepage)
if documents:
    homepage = documents[0]
    print(f"Internal links from {homepage.url}:\n")
    for link in homepage.internal_links[:10]:  # Show first 10
        text_preview = (link.text[:40] + "...") if len(link.text) > 40 else link.text
        print(f"  • {link.url}")
        if text_preview:
            print(f"    → {text_preview}")
    if len(homepage.internal_links) > 10:
        print(f"  ... and {len(homepage.internal_links) - 10} more")

## Domain Analysis

In [ ]:
# Get domains from external links
external_domains = Counter()
for doc in documents:
    for link in doc.external_links:
        try:
            external_domains[link.url.split("/")[2]] += 1  # Extract domain from URL
        except Exception:
            pass

print("Most common external domains:")
for domain, count in external_domains.most_common(10):
    print(f"  {domain}: {count} links")

## Export Data to Multiple Formats

The Serializers module allows exporting crawled documents to JSON, Pandas, Polars, or PyArrow formats. This is useful for data analysis and integration with other tools.

In [ ]:
# Create a serializer instance from the documents
serializer = Serializers(documents)

# Export to JSON
json_file = "cnn_crawl.json"
serializer.to_json(json_file, include_html=False)
print(f"✓ Exported to {json_file}")

# Read and show structure
with open(json_file) as f:
    data = json.load(f)

print("\nJSON structure (first page):")
print(f"  - URL: {data[0]['url']}")
print(f"  - Title: {data[0]['title']}")
print(f"  - Status: {data[0]['status_code']}")
print(f"  - Domain: {data[0]['domain']}")
print(f"  - Internal links: {len(data[0]['internal_links'])}")
print(f"  - External links: {len(data[0]['external_links'])}")

In [ ]:
# Export to Pandas DataFrame with flattened links
df = serializer.to_pandas()

print(f"Pandas DataFrame: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {', '.join(df.columns)}")
print("\nFirst 5 links:")
print(df[["url", "title", "link_url", "link_type"]].head())

In [ ]:
# Analyze link types and domains
print("Link type distribution:")
print(df["link_type"].value_counts())

print("\n\nExternal link domains:")
# Extract domain from URL
df["external_domain"] = df[df["link_type"] == "external"]["link_url"].apply(
    lambda x: x.split("/")[2] if pd.notna(x) else None
)
print(df["external_domain"].value_counts().head(10))

In [ ]:
# Export to Polars (for comparison - often faster for large datasets)
df_polars = serializer.to_polars()

print(f"Polars DataFrame: {df_polars.shape[0]} rows × {df_polars.shape[1]} columns")
print("\nSchema:")
print(df_polars.schema)

In [ ]:
# Export to PyArrow Table for integration with data pipelines
table = serializer.to_arrow()

print(f"PyArrow Table: {table.num_rows} rows × {table.num_columns} columns")
print(f"\nColumn names: {table.column_names}")
print(f"\nDataTypes:\n{table.schema}")